In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.utils import to_categorical
import pandas as pd
import matplotlib.pyplot as plt

# Carga de datos
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

# Procesamiento de df_train
# LLENAR DATOS VACIOS
df_train.loc[df_train['ESTU_VALORMATRICULAUNIVERSIDAD'].isnull(), 'ESTU_VALORMATRICULAUNIVERSIDAD'] = 'No pago matricula'
df_train.loc[df_train['ESTU_HORASSEMANATRABAJA'].isnull(), 'ESTU_HORASSEMANATRABAJA'] = '0'
df_train.loc[df_train['ESTU_PAGOMATRICULAPROPIO'].isnull(), 'ESTU_PAGOMATRICULAPROPIO'] = 'No'
df_train.loc[df_train['FAMI_ESTRATOVIVIENDA'].isnull(), 'FAMI_ESTRATOVIVIENDA'] = 'Sin Estrato'

# Corregir caracteres especiales
df_train['ESTU_PRGM_ACADEMICO'] = df_train['ESTU_PRGM_ACADEMICO'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
df_train['ESTU_PRGM_DEPARTAMENTO'] = df_train['ESTU_PRGM_DEPARTAMENTO'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
df_train['ESTU_VALORMATRICULAUNIVERSIDAD'] = df_train['ESTU_VALORMATRICULAUNIVERSIDAD'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
df_train['ESTU_HORASSEMANATRABAJA'] = df_train['ESTU_HORASSEMANATRABAJA'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')


df_train.drop(columns=['FAMI_TIENEINTERNET.1'], inplace=True)

# Eliminar columnas que no sirven
df_train.drop(columns=['FAMI_TIENELAVADORA', 'FAMI_TIENEAUTOMOVIL', 'ESTU_PRIVADO_LIBERTAD', 'FAMI_EDUCACIONMADRE', 'FAMI_EDUCACIONPADRE'], inplace=True)

# One Hot Encoding para ESTU_VALORMATRICULAUNIVERSIDAD
categorias = ['No pago matricula', 'Menos de 500 mil', 'Entre 500 mil y menos de 1 millon',
              'Entre 1 millon y menos de 2.5 millones', 'Entre 2.5 millones y menos de 4 millones',
              'Entre 4 millones y menos de 5.5 millones', 'Entre 5.5 millones y menos de 7 millones',
              'Mas de 7 millones']

encoder = OneHotEncoder(categories=[categorias])
one_hot_encoded = encoder.fit_transform(df_train[['ESTU_VALORMATRICULAUNIVERSIDAD']])
one_hot_df = pd.DataFrame(one_hot_encoded.toarray(), columns=categorias)
df_train = pd.concat([df_train, one_hot_df], axis=1)
df_train.drop(columns=['ESTU_VALORMATRICULAUNIVERSIDAD'], inplace=True)

# Mapeos
mapeo_estrato = {
    'Sin Estrato': 0,
    'Estrato 1': 1,
    'Estrato 2': 2,
    'Estrato 3': 3,
    'Estrato 4': 4,
    'Estrato 5': 5,
    'Estrato 6': 6
}
mapeo_si_no = {
    'No': 0,
    'Si': 1,
}
df_train['FAMI_ESTRATOVIVIENDA'] = df_train['FAMI_ESTRATOVIVIENDA'].map(mapeo_estrato)
df_train['FAMI_TIENEINTERNET'] = df_train['FAMI_TIENEINTERNET'].map(mapeo_si_no)
df_train['ESTU_PAGOMATRICULAPROPIO'] = df_train['ESTU_PAGOMATRICULAPROPIO'].map(mapeo_si_no)
df_train['FAMI_TIENECOMPUTADOR'] = df_train['FAMI_TIENECOMPUTADOR'].map(mapeo_si_no)

# Codificación de frecuencia para ESTU_PRGM_ACADEMICO
df_train['frecuencia_ESTU_PRGM_ACADEMICO'] = df_train['ESTU_PRGM_ACADEMICO'].map(df_train['ESTU_PRGM_ACADEMICO'].value_counts())
df_train.drop(columns=['ESTU_PRGM_ACADEMICO'], inplace=True)

# One-hot encoding para ESTU_PRGM_DEPARTAMENTO
dummies = df_train['ESTU_PRGM_DEPARTAMENTO'].str.get_dummies()
df_train = pd.concat([df_train, dummies], axis=1)
df_train.drop(columns=['ESTU_PRGM_DEPARTAMENTO'], inplace=True)

# Convertir RENDIMIENTO_GLOBAL a categórico
dummies = df_train['RENDIMIENTO_GLOBAL'].str.get_dummies()
df_train = pd.concat([df_train, dummies], axis=1)
df_train.drop(columns=['RENDIMIENTO_GLOBAL'], inplace=True)
df_train.drop(columns=['PERIODO'], inplace=True)

# Mapeo para ESTU_HORASSEMANATRABAJA
mapeo_horas = {
    '0': 0,
    'Menos de 10 horas': 1,
    'Entre 11 y 20 horas': 2,
    'Entre 21 y 30 horas': 3,
    'Mas de 30 horas': 4
}
df_train['ESTU_HORASSEMANATRABAJA'] = df_train['ESTU_HORASSEMANATRABAJA'].map(mapeo_horas)
df_train.to_csv('train_modified.csv', index=False)

# Procesamiento de df_test
df_test.loc[df_test['ESTU_VALORMATRICULAUNIVERSIDAD'].isnull(), 'ESTU_VALORMATRICULAUNIVERSIDAD'] = 'No pago matricula'
df_test.loc[df_test['ESTU_HORASSEMANATRABAJA'].isnull(), 'ESTU_HORASSEMANATRABAJA'] = '0'
df_test.loc[df_test['ESTU_PAGOMATRICULAPROPIO'].isnull(), 'ESTU_PAGOMATRICULAPROPIO'] = 'No'
df_test.loc[df_test['FAMI_ESTRATOVIVIENDA'].isnull(), 'FAMI_ESTRATOVIVIENDA'] = 'Sin Estrato'

df_test['ESTU_PRGM_ACADEMICO'] = df_test['ESTU_PRGM_ACADEMICO'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
df_test['ESTU_PRGM_DEPARTAMENTO'] = df_test['ESTU_PRGM_DEPARTAMENTO'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
df_test['ESTU_VALORMATRICULAUNIVERSIDAD'] = df_test['ESTU_VALORMATRICULAUNIVERSIDAD'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
df_test['ESTU_HORASSEMANATRABAJA'] = df_test['ESTU_HORASSEMANATRABAJA'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')


df_test.drop(columns=['FAMI_TIENEINTERNET.1'], inplace=True)

df_test.drop(columns=['FAMI_TIENELAVADORA', 'FAMI_TIENEAUTOMOVIL', 'ESTU_PRIVADO_LIBERTAD', 'FAMI_EDUCACIONMADRE', 'FAMI_EDUCACIONPADRE'], inplace=True)

one_hot_encoded = encoder.fit_transform(df_test[['ESTU_VALORMATRICULAUNIVERSIDAD']])
one_hot_df = pd.DataFrame(one_hot_encoded.toarray(), columns=categorias)
df_test = pd.concat([df_test, one_hot_df], axis=1)
df_test.drop(columns=['ESTU_VALORMATRICULAUNIVERSIDAD'], inplace=True)

df_test['FAMI_ESTRATOVIVIENDA'] = df_test['FAMI_ESTRATOVIVIENDA'].map(mapeo_estrato)
df_test['FAMI_TIENEINTERNET'] = df_test['FAMI_TIENEINTERNET'].map(mapeo_si_no)
df_test['ESTU_PAGOMATRICULAPROPIO'] = df_test['ESTU_PAGOMATRICULAPROPIO'].map(mapeo_si_no)
df_test['FAMI_TIENECOMPUTADOR'] = df_test['FAMI_TIENECOMPUTADOR'].map(mapeo_si_no)

df_test['frecuencia_ESTU_PRGM_ACADEMICO'] = df_test['ESTU_PRGM_ACADEMICO'].map(df_test['ESTU_PRGM_ACADEMICO'].value_counts())
df_test.drop(columns=['ESTU_PRGM_ACADEMICO'], inplace=True)

dummies = df_test['ESTU_PRGM_DEPARTAMENTO'].str.get_dummies()
df_test = pd.concat([df_test, dummies], axis=1)
df_test.drop(columns=['ESTU_PRGM_DEPARTAMENTO'], inplace=True)

df_test.drop(columns=['PERIODO'], inplace=True)

df_test['ESTU_HORASSEMANATRABAJA'] = df_test['ESTU_HORASSEMANATRABAJA'].map(mapeo_horas)
df_test.to_csv('test_modified.csv', index=False)




In [1]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


df_train = pd.read_csv('train_modified.csv')
df_test = pd.read_csv('test_modified.csv')

# Suponiendo que las columnas de rendimiento son 'alto', 'bajo', 'medio-alto', 'medio-bajo'
y_train = df_train[['alto', 'bajo', 'medio-alto', 'medio-bajo']]
X_train = df_train.drop(columns=['alto', 'bajo', 'medio-alto', 'medio-bajo'])


# Necesitamos una columna con la etiqueta en forma de string
y_train_labels = y_train.idxmax(axis=1)

model = RandomForestClassifier(n_estimators=1, random_state=30)
model.fit(X_train, y_train_labels)

X_test = df_test.copy()
y_pred = model.predict(X_test)

df_test_ids = pd.read_csv('test_modified.csv')[['ID']]  # Suponiendo que 'ID' es la columna de identificación
df_results = pd.DataFrame({'ID': df_test_ids['ID'], 'RENDIMIENTO_GLOBAL': y_pred})
df_results.to_csv('predicciones.csv', index=False)




NameError: name 'pd' is not defined